# Task 01 | Nested Transaction Analyzer

Concepts: Data structures, sorting, comprehensions
Problem: Given a list of transaction dictionaries with keys id, customer, category, amount, and status, produce a report containing each customer’s successful spending, top category, and rank by spending. Ignore failed transactions and reject duplicate transaction IDs.
Requirements: Return a list sorted by spending descending, then customer name ascending. Customers with no successful transactions must appear with total 0. Do not use pandas.


In [3]:
"""Task 01 - Nested Transaction Analyzer"""

REQUIRED_KEYS = {"id", "customer", "category", "amount", "status"}


def analyze_transactions(transactions):
    """Return a report: customer, total successful spending, top category, rank."""
    if not isinstance(transactions, list):
        raise TypeError("transactions must be a list of dictionaries")

    seen_ids = set()
    totals = {}        # customer -> total successful spending
    by_category = {}   # customer -> {category: amount}

    for t in transactions:
        if not isinstance(t, dict) or not REQUIRED_KEYS <= t.keys():
            raise ValueError(f"Invalid transaction: {t!r}")
        if t["id"] in seen_ids:
            raise ValueError(f"Duplicate transaction id: {t['id']}")
        seen_ids.add(t["id"])

        customer = t["customer"]
        # every customer appears in the report, even with only failed transactions
        totals.setdefault(customer, 0)
        by_category.setdefault(customer, {})

        if t["status"] != "success":
            continue  # ignore failed transactions

        amount = t["amount"]
        if isinstance(amount, bool) or not isinstance(amount, (int, float)) or amount < 0:
            raise ValueError(f"Invalid amount in transaction {t['id']}: {amount!r}")

        totals[customer] += amount
        cats = by_category[customer]
        cats[t["category"]] = cats.get(t["category"], 0) + amount

    report = []
    for customer, total in totals.items():
        cats = by_category[customer]
        # highest amount wins; ties -> alphabetical category
        top = min(cats, key=lambda c: (-cats[c], c)) if cats else None
        report.append({"customer": customer, "total": total, "top_category": top})

    # spending descending, then customer name ascending
    report.sort(key=lambda r: (-r["total"], r["customer"]))

    # rank (equal spending shares the same rank)
    rank, previous_total = 0, None
    for position, row in enumerate(report, start=1):
        if row["total"] != previous_total:
            rank, previous_total = position, row["total"]
        row["rank"] = rank
    return report


if __name__ == "__main__":
    data = [
        {"id": 1, "customer": "Ali", "category": "Food", "amount": 500, "status": "success"},
        {"id": 2, "customer": "Ali", "category": "Travel", "amount": 200, "status": "failed"},
        {"id": 3, "customer": "Sara", "category": "Food", "amount": 700, "status": "success"},
    ]
    for row in analyze_transactions(data):
        print(row)
    try:
        analyze_transactions(data + [data[0]])
    except ValueError as err:
        print("Rejected:", err)


{'customer': 'Sara', 'total': 700, 'top_category': 'Food', 'rank': 1}
{'customer': 'Ali', 'total': 500, 'top_category': 'Food', 'rank': 2}
Rejected: Duplicate transaction id: 1


# Task 02 | Longest Valid Window
Concepts: Sliding window, dictionaries
Problem: Write longest_unique_window(text) to find the longest contiguous substring with no repeated characters. Return the substring, start index, and length.
Requirements: Target O(n) time. On equal-length windows, select the earliest. Treat spaces and punctuation as characters.


In [4]:
def longest_unique_window(text):
    if type(text) != str:
        raise ValueError("text must be a string")

    last_seen = {}       # character -> last index where we saw it
    window_start = 0
    best_start = 0
    best_length = 0

    for i in range(len(text)):
        ch = text[i]
        # if ch is already inside the current window, move the start past it
        if ch in last_seen and last_seen[ch] >= window_start:
            window_start = last_seen[ch] + 1
        last_seen[ch] = i

        length = i - window_start + 1
        if length > best_length:          # '>' keeps the earliest window on ties
            best_length = length
            best_start = window_start

    return text[best_start:best_start + best_length], best_start, best_length


if __name__ == "__main__":
    for sample in ["abcabcbb", "", "bbbbb", "pwwkew", "a b!a b"]:
        print(repr(sample), "->", longest_unique_window(sample))


'abcabcbb' -> ('abc', 0, 3)
'' -> ('', 0, 0)
'bbbbb' -> ('b', 0, 1)
'pwwkew' -> ('wke', 2, 3)
'a b!a b' -> ('a b!', 0, 4)


# Task 03 - Recursive JSON Path Explorer
Techniques: functions, recursion, dict .items(), range(), f-strings (chapters 2-3).

In [5]:
def same_value(value, target):
    if type(value) == bool or type(target) == bool:
        return type(value) == type(target) and value == target
    return value == target


def find_paths(data, target, prefix=""):
    paths = []

    if prefix != "" and same_value(data, target):
        paths.append(prefix)

    if type(data) == dict:
        for key, value in data.items():
            if prefix == "":
                new_path = f"{key}"
            else:
                new_path = f"{prefix}.{key}"
            paths = paths + find_paths(value, target, new_path)
    elif type(data) == list:
        for index in range(len(data)):
            paths = paths + find_paths(data[index], target, f"{prefix}[{index}]")

    return paths


if __name__ == "__main__":
    sample = {"a": [{"x": 5}, 5], "b": {"y": 5}}
    print(find_paths(sample, 5))                  # ['a[0].x', 'a[1]', 'b.y']
    print(find_paths({"p": True, "q": 1}, 1))     # ['q']  (True is not 1)
    print(find_paths({"p": True, "q": 1}, True))  # ['p']
    print(find_paths({}, 5))                      # []


['a[0].x', 'a[1]', 'b.y']
['q']
['p']
[]


# Task 04 - Dependency Execution Planner
Techniques: lists, dictionaries, while loop, min(), tuples (chapters 1-3).

In [6]:
def plan_execution(tasks, dependencies):
    for task in tasks:
        if tasks.count(task) > 1:
            raise ValueError(f"Duplicate task name: {task}")

    waiting_on = {}    # task -> number of unfinished prerequisites
    followers = {}     # task -> list of tasks that wait for it
    for task in tasks:
        waiting_on[task] = 0
        followers[task] = []

    for pair in dependencies:
        first = pair[0]
        second = pair[1]
        if first not in waiting_on:
            raise ValueError(f"Unknown task name: {first}")
        if second not in waiting_on:
            raise ValueError(f"Unknown task name: {second}")
        if first == second:
            raise ValueError(f"Cycle detected: {first} depends on itself")
        if second not in followers[first]:
            followers[first].append(second)
            waiting_on[second] += 1

    available = []
    for task in tasks:
        if waiting_on[task] == 0:
            available.append(task)

    order = []
    while len(available) > 0:
        current = min(available)          # alphabetically first
        available.remove(current)
        order.append(current)
        for next_task in followers[current]:
            waiting_on[next_task] -= 1
            if waiting_on[next_task] == 0:
                available.append(next_task)

    if len(order) != len(tasks):
        stuck = []
        for task in tasks:
            if task not in order:
                stuck.append(task)
        raise ValueError(f"Cycle detected involving: {', '.join(stuck)}")
    return order


if __name__ == "__main__":
    print(plan_execution(["build", "test", "deploy"],
                         [("build", "test"), ("test", "deploy")]))
    print(plan_execution(["c", "b", "a"], []))
    for bad in [[("a", "b"), ("b", "a")], [("a", "zzz")]]:
        try:
            plan_execution(["a", "b"], bad)
        except ValueError as error:
            print("Error:", error)


['build', 'test', 'deploy']
['a', 'b', 'c']
Error: Cycle detected involving: a, b
Error: Unknown task name: zzz


# Task 05 - Expression Validator and Evaluator
Techniques: lists used as stacks (append/pop), while loops, string slicing, if/elif (chapters 1-3).

In [12]:
PRECEDENCE = {"+": 1, "-": 1, "*": 2, "/": 2, "neg": 3}    # "neg" = unary minus
DIGITS = "0123456789"


def tokenize(expression):
    tokens = []
    i = 0
    while i < len(expression):
        ch = expression[i]
        if ch == " " or ch == "\t":
            i += 1
        elif ch in DIGITS:
            j = i
            while j < len(expression) and expression[j] in DIGITS:
                j += 1
            tokens.append(int(expression[i:j]))
            i = j
        elif ch in "+-*/()":
            tokens.append(ch)
            i += 1
        else:
            raise ValueError(f"Invalid character '{ch}' at position {i}")
    return tokens


def apply_operator(operator, values):
    if operator == "neg":
        values.append(-values.pop())
        return
    right = values.pop()
    left = values.pop()
    if operator == "+":
        values.append(left + right)
    elif operator == "-":
        values.append(left - right)
    elif operator == "*":
        values.append(left * right)
    else:
        if right == 0:
            raise ZeroDivisionError("Division by zero")
        values.append(left / right)


def evaluate(expression):
    """Evaluate an integer expression with + - * / ( ) and unary minus (no eval())."""
    tokens = tokenize(expression)
    if len(tokens) == 0:
        raise ValueError("Empty expression")

    values = []          # number stack
    operators = []       # operator stack
    expect_number = True

    for token in tokens:
        if type(token) == int:
            if not expect_number:
                raise ValueError("Missing operator before a number")
            values.append(token)
            expect_number = False
        elif token == "(":
            if not expect_number:
                raise ValueError("Missing operator before '('")
            operators.append("(")
        elif token == ")":
            if expect_number:
                raise ValueError("Unexpected ')'")
            while len(operators) > 0 and operators[-1] != "(":
                apply_operator(operators.pop(), values)
            if len(operators) == 0:
                raise ValueError("Unbalanced parentheses")
            operators.pop()
        elif expect_number:                  # a sign in front of a number
            if token == "-":
                operators.append("neg")
            elif token != "+":
                raise ValueError(f"Unexpected operator '{token}'")
        else:                                # normal operator between two numbers
            while (len(operators) > 0 and operators[-1] != "("
                   and PRECEDENCE[operators[-1]] >= PRECEDENCE[token]):
                apply_operator(operators.pop(), values)
            operators.append(token)
            expect_number = True

    if expect_number:
        raise ValueError("Expression ends unexpectedly")
    while len(operators) > 0:
        operator = operators.pop()
        if operator == "(":
            raise ValueError("Unbalanced parentheses")
        apply_operator(operator, values)
    return values[0]


if __name__ == "__main__":
    tests = ["-(3 + 2) * 4 + 18 / 3", "2 + 3 * 4", "(2 + 3) * 4", "2 * -3",
             "10 / 0", "2 +", "(1 + 2", "1 + 2)", "3 $ 4", "", "4 5", "()"]
    for text in tests:
        try:
            print(f"{text!r:26} = {evaluate(text)}")
        except (ValueError, ZeroDivisionError) as error:
            print(f"{text!r:26} error: {error}")


'-(3 + 2) * 4 + 18 / 3'    = -14.0
'2 + 3 * 4'                = 14
'(2 + 3) * 4'              = 20
'2 * -3'                   = -6
'10 / 0'                   error: Division by zero
'2 +'                      error: Expression ends unexpectedly
'(1 + 2'                   error: Unbalanced parentheses
'1 + 2)'                   error: Unbalanced parentheses
'3 $ 4'                    error: Invalid character '$' at position 2
''                         error: Empty expression
'4 5'                      error: Missing operator before a number
'()'                       error: Unexpected ')'


# Task 06 - Banking Ledger with OOP
Techniques: classes, __init__, private attributes (__balance), tuples (immutable), f-strings (chapters 1-3).

In [ ]:
def check_amount(amount):
    """Reject amounts that are not a positive number."""
    if type(amount) != int and type(amount) != float:
        raise ValueError(f"Amount must be a number, got {amount}")
    if amount <= 0:
        raise ValueError(f"Amount must be positive, got {amount}")


class Account:
    next_transaction_id = 1      # shared counter so every transaction ID is unique

    def __init__(self, account_id, owner, balance=0):
        if balance < 0:
            raise ValueError("Opening balance cannot be negative")
        self.account_id = account_id
        self.owner = owner
        self.__balance = 0
        self.__history = []      # holds tuples: (id, action, amount, balance_after)
        if balance > 0:
            self.deposit(balance, "opening")

    def get_balance(self):
        return self.__balance

    def __record(self, action, amount):
        record = (Account.next_transaction_id, action, amount, self.__balance)   # tuple = immutable
        Account.next_transaction_id += 1
        self.__history.append(record)

    def deposit(self, amount, action="deposit"):
        check_amount(amount)
        self.__balance += amount
        self.__record(action, amount)

    def withdraw(self, amount, action="withdraw"):
        check_amount(amount)
        if amount > self.__balance:
            raise ValueError(f"Insufficient funds in {self.account_id}: "
                             f"balance {self.__balance}, requested {amount}")
        self.__balance -= amount
        self.__record(action, amount)

    def statement(self):
        """Return a copy of the transaction records."""
        return list(self.__history)


class Bank:
    def __init__(self):
        self.accounts = {}

    def open_account(self, account_id, owner, balance=0):
        if account_id in self.accounts:
            raise ValueError(f"Account {account_id} already exists")
        self.accounts[account_id] = Account(account_id, owner, balance)

    def get_account(self, account_id):
        if account_id not in self.accounts:
            raise ValueError(f"Account {account_id} does not exist")
        return self.accounts[account_id]

    def deposit(self, account_id, amount):
        self.get_account(account_id).deposit(amount)

    def withdraw(self, account_id, amount):
        self.get_account(account_id).withdraw(amount)

    def transfer(self, from_id, to_id, amount):
        """Atomic: check everything first, then change both balances."""
        source = self.get_account(from_id)
        target = self.get_account(to_id)
        check_amount(amount)
        if from_id == to_id:
            raise ValueError("Cannot transfer to the same account")
        if amount > source.get_balance():
            raise ValueError(f"Insufficient funds in {from_id}")
        source.withdraw(amount, "transfer_out")
        target.deposit(amount, "transfer_in")


if __name__ == "__main__":
    bank = Bank()
    bank.open_account("A", "Ali", 1000)
    bank.open_account("B", "Sara", 200)
    bank.transfer("A", "B", 350)
    print("A =", bank.get_account("A").get_balance(), "| B =", bank.get_account("B").get_balance())

    for from_id, to_id, amount in [("A", "B", 5000), ("A", "Z", 10), ("A", "B", -5)]:
        try:
            bank.transfer(from_id, to_id, amount)
        except ValueError as error:
            print("Rejected:", error)
    print("After failures -> A =", bank.get_account("A").get_balance(),
          "| B =", bank.get_account("B").get_balance())

    print("Statement for A:")
    for record in bank.get_account("A").statement():
        print("  ", record)


A = 650 | B = 550
Rejected: Insufficient funds in A
Rejected: Account Z does not exist
Rejected: Amount must be positive, got -5
After failures -> A = 650 | B = 550
Statement for A:
   (1, 'opening', 1000, 1000)
   (3, 'transfer_out', 350, 650)


# Task 07 - Employee Payroll Engine
Techniques: abstract class (ABC), inheritance, method overriding, polymorphism (chapter 3).

In [ ]:
from abc import ABC, abstractmethod


def check_not_negative(value, label):
    if value < 0:
        raise ValueError(f"{label} cannot be negative, got {value}")
    return value


class Employee(ABC):

    def __init__(self, name):
        if name.strip() == "":
            raise ValueError("Employee name cannot be empty")
        self.name = name.strip()

    @abstractmethod
    def calculate_pay(self):
        """"""
class SalariedEmployee(Employee):
    def __init__(self, name, salary):
        super().__init__(name)
        self.salary = check_not_negative(salary, "Salary")

    def calculate_pay(self):
        return self.salary


class HourlyEmployee(Employee):
    def __init__(self, name, hourly_rate, hours_worked):
        super().__init__(name)
        self.hourly_rate = check_not_negative(hourly_rate, "Hourly rate")
        self.hours_worked = check_not_negative(hours_worked, "Hours worked")

    def calculate_pay(self):
        if self.hours_worked <= 40:
            return self.hours_worked * self.hourly_rate
        overtime_hours = self.hours_worked - 40
        return 40 * self.hourly_rate + overtime_hours * self.hourly_rate * 1.5


class CommissionEmployee(Employee):
    def __init__(self, name, base_pay, sales, rate):
        super().__init__(name)
        self.base_pay = check_not_negative(base_pay, "Base pay")
        self.sales = check_not_negative(sales, "Sales")
        self.rate = check_not_negative(rate, "Commission rate")

    def calculate_pay(self):
        return self.base_pay + self.sales * self.rate


def payroll_summary(employees):
    total = 0
    for employee in employees:
        pay = employee.calculate_pay()          # polymorphism: same call, different formula
        print(f"{employee.name:8} {employee.__class__.__name__:20} {pay:>10,.2f}")
        total += pay
    print(f"{'TOTAL':29} {total:>10,.2f}")
    return total


if __name__ == "__main__":
    try:
        Employee("Nobody")
    except TypeError as error:
        print("Cannot create Employee:", error)

    staff = [SalariedEmployee("Ayesha", 3000),
             HourlyEmployee("Bilal", 20, 45),            # 950
             CommissionEmployee("Chand", 500, 4000, 0.10)]   # 900
    payroll_summary(staff)


Cannot create Employee: Can't instantiate abstract class Employee without an implementation for abstract method 'calculate_pay'
Ayesha   SalariedEmployee       3,000.00
Bilal    HourlyEmployee           950.00
Chand    CommissionEmployee       900.00
TOTAL                           4,850.00


# Task 08 - Cache Decorator with Expiration
Techniques: functions inside functions (closures), dictionaries, tuples, .get() (chapters 1-3).
Needs two things the chapters do not cover: *args/**kwargs and the @decorator syntax.

Behavior notes
- Keyword arguments are sorted, so f(x=2, y=3) and f(y=3, x=2) use the same cache entry.
- Unhashable arguments (lists, dicts) cannot be cache keys: the function is simply called
  normally every time and each call counts as a miss.
- Time is measured with time.monotonic().

In [ ]:
import functools
import time


def ttl_cache(seconds):
    if seconds <= 0:
        raise ValueError("seconds must be positive")

    def decorator(func):
        cache = {}                                  # key -> (result, expiry_time)
        stats = {"hits": 0, "misses": 0}

        @functools.wraps(func)                      # keeps the function's name and docstring
        def wrapper(*args, **kwargs):
            key = (args, tuple(sorted(kwargs.items())))
            try:
                hash(key)
            except TypeError:                       # unhashable input -> no caching
                stats["misses"] += 1
                return func(*args, **kwargs)

            if key in cache and time.monotonic() < cache[key][1]:
                stats["hits"] += 1
                return cache[key][0]

            stats["misses"] += 1
            result = func(*args, **kwargs)
            cache[key] = (result, time.monotonic() + seconds)
            return result

        def cache_info():
            return {"hits": stats["hits"], "misses": stats["misses"], "size": len(cache)}

        wrapper.cache_info = cache_info
        return wrapper

    return decorator


if __name__ == "__main__":
    @ttl_cache(0.3)
    def add(x, y):
        """Add two numbers."""
        return x + y

    print(add(x=2, y=3), add(y=3, x=2))     # second call is a hit
    print(add.cache_info())                  # hits=1, misses=1
    time.sleep(0.4)
    add(x=2, y=3)                            # expired -> miss
    print(add.cache_info())                  # hits=1, misses=2

    @ttl_cache(1)
    def total(items):
        return sum(items)

    print(total([1, 2, 3]), total([1, 2, 3]), total.cache_info())   # lists: never cached
    print(add.__name__, "-", add.__doc__)


5 5
{'hits': 1, 'misses': 1, 'size': 1}
{'hits': 1, 'misses': 2, 'size': 1}
6 6 {'hits': 0, 'misses': 2, 'size': 0}
add - Add two numbers.


# Task 09 - Streaming Log Inspector
Techniques: for loop over a file, .split(), .strip(), slicing, dictionaries with .get() (chapter 2-3).
Needs `yield` (generator) and open(), which the chapters do not cover.
Line format: YYYY-MM-DD HH:MM:SS | LEVEL | message

In [13]:
def is_valid_timestamp(text):
    if len(text) != 19:
        return False
    if text[4] != "-" or text[7] != "-" or text[10] != " " or text[13] != ":" or text[16] != ":":
        return False
    numbers = text[0:4] + text[5:7] + text[8:10] + text[11:13] + text[14:16] + text[17:19]
    return numbers.isdigit()


def parse_line(line):
    parts = line.split("|", 2)               # max 2 splits so '|' can appear in the message
    if len(parts) != 3:
        return None
    timestamp = parts[0].strip()
    level = parts[1].strip()
    message = parts[2].strip()
    if not is_valid_timestamp(timestamp) or level == "" or message == "":
        return None
    return timestamp, level, message


def read_errors(path, stats=None):
    if stats is None:
        stats = {}
    stats["malformed"] = 0

    with open(path, encoding="utf-8") as file:
        line_number = 0
        for line in file:
            line_number += 1
            line = line.strip()
            if line == "":
                continue
            parsed = parse_line(line)
            if parsed is None:
                stats["malformed"] += 1
                continue
            if parsed[1] == "ERROR":
                yield line_number, parsed[0], parsed[2]


def summarize_errors(path):
    stats = {}
    counts = {}
    for line_number, timestamp, message in read_errors(path, stats):
        date = timestamp[:10]
        counts[date] = counts.get(date, 0) + 1
    return counts, stats["malformed"]


def create_sample_log(path="sample.log"):
    with open(path, "w", encoding="utf-8") as file:
        file.write("2026-11-07 10:30:00 | ERROR | Connection lost\n")
        file.write("2026-11-07 10:31:00 | INFO | Reconnected\n")
        file.write("this line is broken\n")
        file.write("2026-11-07 11:00:00 | ERROR | Disk almost full | check /var\n")
        file.write("2026-11-08 09:15:00 | ERROR | Timeout\n")
        file.write("2026-1x-08 09:20:00 | ERROR | bad timestamp\n")
        file.write("2026-11-08 09:20:00 | WARNING | Slow response\n")
        file.write("\n")
        file.write("2026-11-08 12:00:00 | ERROR | Service crashed\n")
    return path


if __name__ == "__main__":
    log_path = create_sample_log()
    stats = {}
    for item in read_errors(log_path, stats):
        print(item)
    print("Malformed lines:", stats["malformed"])
    print("Errors per date:", summarize_errors(log_path))


(1, '2026-11-07 10:30:00', 'Connection lost')
(4, '2026-11-07 11:00:00', 'Disk almost full | check /var')
(5, '2026-11-08 09:15:00', 'Timeout')
(9, '2026-11-08 12:00:00', 'Service crashed')
Malformed lines: 2
Errors per date: ({'2026-11-07': 2, '2026-11-08': 2}, 2)


# Task 10 - CSV Inventory Reconciliation
Techniques: .split(","), .strip(), .upper(), dictionaries, lists, f-strings (chapters 1-3).
Needs open() to read/write files, which the chapters do not cover.

In [14]:
def read_rows(path):
    """Read a CSV file and return a list of rows (each row is a list of text values)."""
    rows = []
    with open(path, encoding="utf-8") as file:
        first_line = True
        for line in file:
            line = line.strip()
            if first_line:               # skip the header
                first_line = False
                continue
            if line == "":
                continue
            parts = line.split(",")
            clean = []
            for part in parts:
                clean.append(part.strip())
            rows.append(clean)
    return rows


def load_inventory(path):
    """inventory.csv -> {sku: [name, quantity]}"""
    inventory = {}
    for row in read_rows(path):
        if len(row) != 3 or not row[2].isdigit() or row[0] == "" or row[0] in inventory:
            raise ValueError(f"Bad inventory row: {row}")
        inventory[row[0]] = [row[1], int(row[2])]
    return inventory


def apply_movements(inventory, path):
    """Apply IN / OUT movements in order. Returns the list of rejected rows."""
    rejected = []
    for row in read_rows(path):
        if len(row) != 3:
            rejected.append([",".join(row), "", "", "wrong number of columns"])
            continue
        sku = row[0]
        kind = row[1].upper()
        units_text = row[2]

        reason = ""
        units = 0
        if sku not in inventory:
            reason = "unknown SKU"
        elif kind != "IN" and kind != "OUT":
            reason = "invalid movement type"
        else:
            # accept an optional leading '-' so negative numbers are reported properly
            digits = units_text[1:] if units_text.startswith("-") else units_text
            if not digits.isdigit():
                reason = "units is not a whole number"
            else:
                units = int(units_text)
                if units <= 0:
                    reason = "units must be positive"
                elif kind == "OUT" and units > inventory[sku][1]:
                    reason = "insufficient stock (would go negative)"

        if reason != "":
            rejected.append([sku, row[1], units_text, reason])
            continue                      # keep going with the next row

        if kind == "IN":
            inventory[sku][1] += units
        else:
            inventory[sku][1] -= units
    return rejected


def write_outputs(inventory, rejected):
    with open("updated_inventory.csv", "w", encoding="utf-8") as file:
        file.write("sku,name,quantity\n")
        for sku, item in inventory.items():
            file.write(f"{sku},{item[0]},{item[1]}\n")
    with open("rejected_movements.csv", "w", encoding="utf-8") as file:
        file.write("sku,type,units,reason\n")
        for row in rejected:
            file.write(f"{row[0]},{row[1]},{row[2]},{row[3]}\n")


def create_sample_data():
    with open("inventory.csv", "w", encoding="utf-8") as file:
        file.write("sku,name,quantity\nP01,Widget,10\nP02,Gadget,5\n")
    with open("movements.csv", "w", encoding="utf-8") as file:
        file.write("sku,type,units\n"
                   "P01,OUT,4\nP01,OUT,9\nP01,IN,2\n"
                   "P02,IN,0\nP02,IN,-3\nP99,IN,3\nP02,MOVE,1\nP02,OUT,abc\nP02,OUT,5\n")


def main():
    create_sample_data()
    inventory = load_inventory("inventory.csv")
    rejected = apply_movements(inventory, "movements.csv")
    write_outputs(inventory, rejected)
    for sku, item in inventory.items():
        print(sku, item[0], item[1])
    print("Rejected rows:")
    for row in rejected:
        print("  ", row)


if __name__ == "__main__":
    main()

P01 Widget 8
P02 Gadget 0
Rejected rows:
   ['P01', 'OUT', '9', 'insufficient stock (would go negative)']
   ['P02', 'IN', '0', 'units must be positive']
   ['P02', 'IN', '-3', 'units must be positive']
   ['P99', 'IN', '3', 'unknown SKU']
   ['P02', 'MOVE', '1', 'invalid movement type']
   ['P02', 'OUT', 'abc', 'units is not a whole number']


# Task 11 - Multi-File Student Gradebook
Techniques: dictionaries, .split(","), loops, f-strings, functions with return (chapters 1-3).
Needs open() and try/except (for float()), which the chapters do not cover.

In [15]:
def read_rows(path):
    rows = []
    with open(path, encoding="utf-8") as file:
        first_line = True
        for line in file:
            line = line.strip()
            if first_line:
                first_line = False
                continue
            if line == "":
                continue
            row = []
            for part in line.split(","):
                row.append(part.strip())
            rows.append(row)
    return rows


def get_grade(average):
    if average >= 85:
        return "A"
    elif average >= 70:
        return "B"
    elif average >= 55:
        return "C"
    elif average >= 40:
        return "D"
    return "F"


def load_students(path):
    """students.csv -> {student_id: name}"""
    students = {}
    for row in read_rows(path):
        if len(row) != 2 or row[0] == "" or row[1] == "":
            raise ValueError(f"Bad student row: {row}")
        if row[0] in students:
            raise ValueError(f"Duplicate student id: {row[0]}")
        students[row[0]] = row[1]
    return students


def load_scores(path, students):
    """scores.csv -> ({student_id: {subject: marks}}, rejected_rows)"""
    scores = {}
    rejected = []
    for row in read_rows(path):
        if len(row) != 3:
            rejected.append((row, "wrong number of columns"))
            continue
        student_id = row[0]
        subject = row[1]
        try:
            marks = float(row[2])
        except ValueError:
            rejected.append((row, "marks is not a number"))
            continue

        if student_id not in students:
            rejected.append((row, "unknown student id"))
        elif marks < 0 or marks > 100:
            rejected.append((row, "marks must be between 0 and 100"))
        elif subject in scores.get(student_id, {}):
            rejected.append((row, "duplicate student-subject entry"))
        else:
            if student_id not in scores:
                scores[student_id] = {}
            scores[student_id][subject] = marks
    return scores, rejected


def build_report(students, scores):
    report = []
    for student_id, subjects in scores.items():
        total = 0
        for marks in subjects.values():
            total += marks
        average = total / len(subjects)
        report.append({"student_id": student_id, "name": students[student_id],
                       "average": average})

    # bubble sort: higher average first, then smaller ID first
    count = len(report)
    for i in range(count):
        for j in range(count - i - 1):
            a = report[j]
            b = report[j + 1]
            if a["average"] < b["average"] or (a["average"] == b["average"]
                                               and a["student_id"] > b["student_id"]):
                report[j] = b
                report[j + 1] = a

    rank = 1
    for row in report:
        row["rank"] = rank
        row["grade"] = get_grade(row["average"])
        row["average"] = round(row["average"], 2)
        rank += 1
    return report


def write_report(report, path="report.csv"):
    with open(path, "w", encoding="utf-8") as file:
        file.write("rank,student_id,name,average,grade\n")
        for row in report:
            file.write(f"{row['rank']},{row['student_id']},{row['name']},"
                       f"{row['average']},{row['grade']}\n")


def create_sample_data():
    with open("students.csv", "w", encoding="utf-8") as file:
        file.write("student_id,name\n101,Ali\n102,Sara\n103,Bilal\n")
    with open("scores.csv", "w", encoding="utf-8") as file:
        file.write("student_id,subject,marks\n"
                   "101,Math,90\n101,Physics,80\n101,English,85\n"
                   "102,Math,95\n102,Physics,70\n"
                   "103,Math,35\n103,Math,50\n"
                   "999,Math,80\n"
                   "102,English,120\n")


def main():
    create_sample_data()
    students = load_students("students.csv")
    scores, rejected = load_scores("scores.csv", students)
    report = build_report(students, scores)
    write_report(report)
    for row in report:
        print(row)
    print("Rejected score rows:")
    for row, reason in rejected:
        print("  ", row, "->", reason)


if __name__ == "__main__":
    main()


{'student_id': '101', 'name': 'Ali', 'average': 85.0, 'rank': 1, 'grade': 'A'}
{'student_id': '102', 'name': 'Sara', 'average': 82.5, 'rank': 2, 'grade': 'B'}
{'student_id': '103', 'name': 'Bilal', 'average': 35.0, 'rank': 3, 'grade': 'F'}
Rejected score rows:
   ['103', 'Math', '50'] -> duplicate student-subject entry
   ['999', 'Math', '80'] -> unknown student id
   ['102', 'English', '120'] -> marks must be between 0 and 100


# Task 12 - Custom Iterable Range Toolkit
Techniques: classes, __init__, while loops, // floor division, if/else (chapters 1-3).
Needs the special methods __iter__, __reversed__, __len__, __contains__ and `yield`
(the task itself asks for them).

In [ ]:
class ReversibleRange:
    """Works like range(start, stop, step), but nothing is stored in a list."""

    def __init__(self, start, stop, step=1):
        if type(start) != int or type(stop) != int or type(step) != int:
            raise ValueError("start, stop and step must be integers")
        if step == 0:
            raise ValueError("step cannot be zero")
        self.start = start
        self.stop = stop
        self.step = step

    def __len__(self):
        if self.step > 0:
            distance = self.stop - self.start
            count = (distance + self.step - 1) // self.step
        else:
            distance = self.start - self.stop
            count = (distance - self.step - 1) // (-self.step)
        if count < 0:
            return 0
        return count

    def __iter__(self):
        current = self.start
        if self.step > 0:
            while current < self.stop:
                yield current
                current += self.step
        else:
            while current > self.stop:
                yield current
                current += self.step

    def __reversed__(self):
        index = len(self) - 1
        while index >= 0:                       # last item first
            yield self.start + index * self.step
            index -= 1

    def __contains__(self, value):
        if type(value) != int:
            return False
        if self.step > 0:
            inside = value >= self.start and value < self.stop
        else:
            inside = value <= self.start and value > self.stop
        return inside and (value - self.start) % self.step == 0


if __name__ == "__main__":
    r = ReversibleRange(2, 11, 3)
    print(list(r), list(reversed(r)), len(r), 5 in r, 6 in r)        # [2,5,8] [8,5,2] 3 True False
    n = ReversibleRange(10, 0, -3)
    print(list(n), list(reversed(n)), len(n), 7 in n, 0 in n)        # [10,7,4,1] [1,4,7,10] 4 True False
    print(list(ReversibleRange(5, 5)), len(ReversibleRange(5, 1)))   # [] 0

    # compare with Python's own range
    for a, b, c in [(0, 10, 1), (0, 10, 4), (10, 0, -2), (3, -8, -5), (-5, 5, 3)]:
        mine = ReversibleRange(a, b, c)
        real = range(a, b, c)
        assert list(mine) == list(real)
        assert list(reversed(mine)) == list(reversed(real))
        assert len(mine) == len(real)
        for x in range(-12, 13):
            assert (x in mine) == (x in real)
    print("Matches Python's range")


# Task 13 - Booking Conflict Scheduler
Techniques: classes, dictionaries, lists, loops, tuples (chapters 1-3).
Needs the datetime module (the task asks for datetime objects).

In [ ]:
from datetime import datetime


class Booking:
    def __init__(self, booking_id, room, start, end):
        self.booking_id = booking_id
        self.room = room
        self.start = start
        self.end = end


class Scheduler:
    def __init__(self):
        self.bookings = {}      # room -> list of Booking
        self.used_ids = []

    def book(self, booking_id, room, start, end):
        """Try to book a room.

        Returns:
            (True, [])            if accepted
            (False, [conflicts])  if it overlaps existing bookings in the same room
        Raises:
            ValueError for end <= start or a duplicate booking ID.
        """
        if end <= start:
            raise ValueError("End time must be after start time")
        if booking_id in self.used_ids:
            raise ValueError(f"Duplicate booking id: {booking_id}")

        if room not in self.bookings:
            self.bookings[room] = []

        conflicts = []
        for existing in self.bookings[room]:
            # overlap only when each one starts before the other ends
            # (back-to-back bookings are allowed)
            if start < existing.end and existing.start < end:
                conflicts.append(existing)
        if len(conflicts) > 0:
            return False, conflicts

        self.bookings[room].append(Booking(booking_id, room, start, end))
        self.used_ids.append(booking_id)
        return True, []


if __name__ == "__main__":
    def at(hour, minute=0):
        return datetime(2026, 11, 7, hour, minute)

    scheduler = Scheduler()
    requests = [("B1", "A", at(9), at(10)), ("B2", "A", at(10), at(11)),
                ("B3", "A", at(9, 30), at(10, 30)), ("B4", "B", at(9, 30), at(10, 30)),
                ("B1", "B", at(12), at(13)), ("B5", "A", at(12), at(11))]
    for booking_id, room, start, end in requests:
        try:
            accepted, conflicts = scheduler.book(booking_id, room, start, end)
        except ValueError as error:
            print(booking_id, "invalid:", error)
            continue
        if accepted:
            print(booking_id, "accepted")
        else:
            names = []
            for c in conflicts:
                names.append(c.booking_id)
            print(booking_id, "rejected, conflicts with", names)


# Task 14 - Command-Line Contact Manager
Techniques: functions, dictionaries, lists, while loop with break, string methods, f-strings (chapters 1-3).
Needs the json and os modules and try/except (storage + corrupt-file handling).

Layout: storage functions -> business-logic functions -> menu.

In [ ]:
import json
import os

DATA_FILE = "contacts.json"


# ---------------- storage ----------------
def load_contacts(path=DATA_FILE):
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as file:
        text = file.read()
    try:
        contacts = json.loads(text)
        if type(contacts) != list:
            raise ValueError("file does not contain a list")
        return contacts
    except ValueError as error:
        backup = path + ".corrupt.bak"
        with open(backup, "w", encoding="utf-8") as file:
            file.write(text)
        raise ValueError(f"{path} is corrupt ({error}). A copy was kept in {backup}")


def save_contacts(contacts, path=DATA_FILE):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(contacts, file, indent=2)


# ---------------- business logic ----------------
def check_contact(name, phone, email):
    if name.strip() == "":
        raise ValueError("Name cannot be empty")

    digits = phone.strip().replace("-", "").replace(" ", "")
    if digits.startswith("+"):
        digits = digits[1:]
    if not digits.isdigit() or len(digits) < 7:
        raise ValueError("Phone must have at least 7 digits")

    email = email.strip()
    if email.count("@") != 1 or email.startswith("@") or email.endswith("@"):
        raise ValueError("Email is not valid")
    if "." not in email.split("@")[1]:
        raise ValueError("Email is not valid")


def add_contact(contacts, name, phone, email):
    check_contact(name, phone, email)
    biggest_id = 0
    for contact in contacts:
        if contact["id"] > biggest_id:
            biggest_id = contact["id"]
    new_contact = {"id": biggest_id + 1, "name": name.strip(),
                   "phone": phone.strip(), "email": email.strip()}
    contacts.append(new_contact)
    return new_contact


def search_contacts(contacts, text):
    found = []
    for contact in contacts:
        if text.strip().lower() in contact["name"].lower():     # case-insensitive
            found.append(contact)
    return found


def find_contact(contacts, contact_id):
    for contact in contacts:
        if contact["id"] == contact_id:
            return contact
    raise ValueError(f"No contact with id {contact_id}")


def update_contact(contacts, contact_id, name, phone, email):
    """Blank values keep the old data."""
    contact = find_contact(contacts, contact_id)
    new_name = name if name.strip() != "" else contact["name"]
    new_phone = phone if phone.strip() != "" else contact["phone"]
    new_email = email if email.strip() != "" else contact["email"]
    check_contact(new_name, new_phone, new_email)
    contact["name"] = new_name.strip()
    contact["phone"] = new_phone.strip()
    contact["email"] = new_email.strip()


def delete_contact(contacts, contact_id):
    contact = find_contact(contacts, contact_id)
    contacts.remove(contact)
    return contact


# ---------------- menu ----------------
def show(contacts):
    if len(contacts) == 0:
        print("  (no contacts)")
    for c in contacts:
        print(f"  [{c['id']}] {c['name']} | {c['phone']} | {c['email']}")


def ask_id(prompt):
    text = input(prompt).strip()
    if not text.isdigit():
        raise ValueError("ID must be a number")
    return int(text)


def main(path=DATA_FILE):
    try:
        contacts = load_contacts(path)
    except ValueError as error:
        print("ERROR:", error)
        print("Nothing was changed. Fix or move the file, then start again.")
        return

    while True:
        print("\n1) Add  2) Search  3) Update  4) Delete  5) List  0) Exit")
        choice = input("Choose: ").strip()
        try:
            if choice == "1":
                contact = add_contact(contacts, input("Name: "), input("Phone: "), input("Email: "))
                save_contacts(contacts, path)
                print(f"Added with ID {contact['id']}")
            elif choice == "2":
                show(search_contacts(contacts, input("Search name: ")))
            elif choice == "3":
                contact_id = ask_id("ID to update: ")
                print("(leave blank to keep the old value)")
                update_contact(contacts, contact_id, input("Name: "), input("Phone: "), input("Email: "))
                save_contacts(contacts, path)
                print("Updated.")
            elif choice == "4":
                removed = delete_contact(contacts, ask_id("ID to delete: "))
                save_contacts(contacts, path)
                print("Deleted", removed["name"])
            elif choice == "5":
                show(contacts)
            elif choice == "0":
                print("Goodbye!")
                break
            else:
                print("Invalid choice.")
        except ValueError as error:
            print("Error:", error)


if __name__ == "__main__":
    main()


# Task 15 - Integrated Order Fulfillment System
Techniques: classes working together, dictionaries, lists, loops, f-strings (chapters 1-3).
Needs json, uuid, try/except and unittest (the task asks for JSON files and unit tests).

In [ ]:
import json
import os
import sys
import tempfile
import unittest
import uuid


class Product:
    def __init__(self, product_id, name, price):
        self.product_id = product_id
        self.name = name
        self.price = price


class Inventory:
    def __init__(self):
        self.products = {}     # product_id -> Product
        self.stock = {}        # product_id -> quantity in stock

    def add_product(self, product, quantity):
        if quantity < 0:
            raise ValueError("Stock cannot be negative")
        self.products[product.product_id] = product
        self.stock[product.product_id] = quantity

    def load_catalog(self, path):
        """JSON file: [{"id": "P1", "name": "Pen", "price": 100, "stock": 5}, ...]"""
        with open(path, encoding="utf-8") as file:
            items = json.load(file)
        for item in items:
            self.add_product(Product(item["id"], item["name"], item["price"]), item["stock"])

    def check_stock(self, items):
        """Check EVERY item before anything changes. items = {product_id: quantity}."""
        for product_id, quantity in items.items():
            if product_id not in self.products:
                raise ValueError(f"Unknown product: {product_id}")
            if self.stock[product_id] < quantity:
                raise ValueError(f"Out of stock: {product_id} "
                                 f"(wanted {quantity}, have {self.stock[product_id]})")

    def take_stock(self, items):
        for product_id, quantity in items.items():
            self.stock[product_id] -= quantity

    def put_back_stock(self, items):
        for product_id, quantity in items.items():
            self.stock[product_id] += quantity


class Cart:
    def __init__(self, inventory):
        self.inventory = inventory
        self.items = {}            # product_id -> quantity
        self.discount_percent = 0

    def add_item(self, product_id, quantity):
        if type(quantity) != int or quantity <= 0:
            raise ValueError(f"Quantity must be a positive whole number, got {quantity}")
        if product_id not in self.inventory.products:
            raise ValueError(f"Unknown product: {product_id}")
        self.items[product_id] = self.items.get(product_id, 0) + quantity

    def set_discount(self, percent):
        if percent < 0 or percent > 100:
            raise ValueError("Discount must be between 0 and 100")
        self.discount_percent = percent

    def get_total(self):
        subtotal = 0
        for product_id, quantity in self.items.items():
            subtotal += self.inventory.products[product_id].price * quantity
        return round(subtotal * (100 - self.discount_percent) / 100, 2)


class Order:
    def __init__(self, cart):
        self.order_id = uuid.uuid4().hex[:10].upper()      # unique ID
        self.lines = []
        for product_id, quantity in cart.items.items():
            product = cart.inventory.products[product_id]
            self.lines.append({"product_id": product_id, "name": product.name,
                               "price": product.price, "quantity": quantity})
        self.discount_percent = cart.discount_percent
        self.total = cart.get_total()

    def to_dict(self):
        return {"order_id": self.order_id, "lines": self.lines,
                "discount_percent": self.discount_percent, "total": self.total}


def save_order(order, path):
    """Add the order to the JSON file."""
    orders = []
    if os.path.exists(path):
        with open(path, encoding="utf-8") as file:
            orders = json.load(file)
    orders.append(order.to_dict())
    with open(path, "w", encoding="utf-8") as file:
        json.dump(orders, file, indent=2)


def place_order(cart, inventory, orders_path):
    """Validate -> reduce stock -> save. If saving fails, stock is put back."""
    if len(cart.items) == 0:
        raise ValueError("Cart is empty")

    inventory.check_stock(cart.items)            # 1. validate everything first
    order = Order(cart)
    bought = dict(cart.items)
    inventory.take_stock(bought)                 # 2. change stock
    try:
        save_order(order, orders_path)           # 3. save the order
    except Exception as error:
        inventory.put_back_stock(bought)         # roll back
        raise RuntimeError(f"Order was not saved, stock restored: {error}")

    cart.items = {}
    cart.discount_percent = 0
    return order


# ---------------- unit tests ----------------
class OrderTests(unittest.TestCase):
    def setUp(self):
        self.folder = tempfile.TemporaryDirectory()
        self.orders_path = os.path.join(self.folder.name, "orders.json")
        self.inventory = Inventory()
        self.inventory.add_product(Product("P1", "Notebook", 100), 5)
        self.inventory.add_product(Product("P2", "Pen", 10), 1)

    def tearDown(self):
        self.folder.cleanup()

    def test_order_with_discount(self):
        cart = Cart(self.inventory)
        cart.add_item("P1", 2)
        cart.set_discount(10)
        order = place_order(cart, self.inventory, self.orders_path)
        self.assertEqual(order.total, 180)
        self.assertEqual(self.inventory.stock["P1"], 3)

    def test_out_of_stock_changes_nothing(self):
        cart = Cart(self.inventory)
        cart.add_item("P1", 2)
        cart.add_item("P2", 5)               # only 1 in stock
        with self.assertRaises(ValueError):
            place_order(cart, self.inventory, self.orders_path)
        self.assertEqual(self.inventory.stock["P1"], 5)

    def test_invalid_quantity(self):
        cart = Cart(self.inventory)
        for bad in [0, -1, 1.5, "2"]:
            with self.assertRaises(ValueError):
                cart.add_item("P1", bad)

    def test_rollback_when_saving_fails(self):
        cart = Cart(self.inventory)
        cart.add_item("P1", 2)
        with self.assertRaises(RuntimeError):
            place_order(cart, self.inventory, self.folder.name)   # a folder, not a file
        self.assertEqual(self.inventory.stock["P1"], 5)

    def test_unique_order_ids(self):
        ids = []
        for i in range(3):
            cart = Cart(self.inventory)
            cart.add_item("P1", 1)
            ids.append(place_order(cart, self.inventory, self.orders_path).order_id)
        self.assertEqual(len(set(ids)), 3)


# ---------------- demo ----------------
def demo():
    with open("catalog.json", "w", encoding="utf-8") as file:
        json.dump([{"id": "P1", "name": "Notebook", "price": 100, "stock": 5},
                   {"id": "P2", "name": "Pen", "price": 10, "stock": 20}], file, indent=2)

    inventory = Inventory()
    inventory.load_catalog("catalog.json")
    cart = Cart(inventory)
    cart.add_item("P1", 2)
    cart.set_discount(10)
    order = place_order(cart, inventory, "orders.json")
    print(f"Order {order.order_id} total = {order.total}")      # 180.0
    print("P1 stock now:", inventory.stock["P1"])               # 3

    cart.add_item("P1", 10)
    try:
        place_order(cart, inventory, "orders.json")
    except ValueError as error:
        print("Rejected:", error)


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "test":
        unittest.main(argv=[sys.argv[0]], verbosity=2)
    else:
        demo()
